# 연습문제 3-1 ~ 3-2 풀이: 다층 퍼셉트론의 파라미터 수와 순전파

3-1절 연습문제 3-1 ~ 3-2의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch03/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch
import torch.nn as nn

## 연습문제 3-1

> 3 x 2 크기의 이미지를 입력받아 2 x 3 크기의 이미지를 출력하는 다층 퍼셉트론 모델을 그림으로 그려 보자.
> 은닉층은 두 개이며 뉴런의 수는 각각 10개와 8개다. 이 모델에 포함된 전체 파라미터의 수를 계산해 보자.
>
> 힌트: 이미지의 형태는 무시하고 픽셀의 수로만 생각해 보자.

**구조 정하기**
- 입력 이미지 3 x 2는 픽셀 6개, 출력 이미지 2 x 3도 픽셀 6개: 입력 뉴런 6개, 출력 뉴런 6개
- 그림으로 그리면 6개 -> 10개 -> 8개 -> 6개로 이어지는 완전 연결 구조
- 계층 하나의 파라미터 수는 (입력 수 * 뉴런 수) + 뉴런 수: 가중치는 입력과 뉴런의 모든 쌍마다 하나, 편향은 뉴런마다 하나

In [2]:
sizes = [6, 10, 8, 6]                       # 입력 픽셀 수, 은닉층 두 개, 출력 픽셀 수
total = 0
for n_in, n_out in zip(sizes[:-1], sizes[1:]):
    weights, biases = n_in * n_out, n_out   # 가중치 수와 편향 수
    count = weights + biases
    total += count
    print(f'{n_in:2d} -> {n_out:2d}: 가중치 {weights:2d} + 편향 {biases:2d} = {count}')
print(f'전체 파라미터: {total}')

 6 -> 10: 가중치 60 + 편향 10 = 70
10 ->  8: 가중치 80 + 편향  8 = 88
 8 ->  6: 가중치 48 + 편향  6 = 54
전체 파라미터: 212


**결과 분석: 계층별로 70개, 88개, 54개이고 전체는 212개**
- 첫 번째 계층: 6 * 10 + 10 = 70
- 두 번째 계층: 10 * 8 + 8 = 88
- 세 번째 계층: 8 * 6 + 6 = 54
- 합계는 70 + 88 + 54 = 212개

**파이토치로 확인하기**
- [코드 3-1]과 같은 방식의 모델 클래스로 같은 구조를 만들고, 계층별 파라미터 수를 세어 손으로 계산한 값과 비교
- 시그모이드 활성화 계층은 학습으로 바뀌는 값이 없어 파라미터가 0개
- 이미지 한 장을 한 줄로 펼쳐 넣고, 출력은 다시 2 x 3으로 접어 형태가 맞는지 확인

In [3]:
class MLP(nn.Module):                               # [코드 3-1]과 같은 방식의 모델 클래스
    def __init__(self):
        super().__init__()
        self.hidden_layer1 = nn.Linear(6, 10)       # 은닉층 1: 입력 6개, 뉴런 10개
        self.hidden_layer2 = nn.Linear(10, 8)       # 은닉층 2: 입력 10개, 뉴런 8개
        self.output_layer = nn.Linear(8, 6)         # 출력층: 입력 8개, 뉴런 6개
        self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.hidden_layer1(x))
        x = self.activation(self.hidden_layer2(x))
        return self.output_layer(x)

In [4]:
model = MLP()
layers = (model.hidden_layer1, model.hidden_layer2, model.output_layer, model.activation)
for layer in layers:
    count = sum(p.numel() for p in layer.parameters())
    print(f'{layer.__class__.__name__:8} {count:3d}개')
print('전체:', sum(p.numel() for p in model.parameters()))

image = torch.randn(1, 3, 2)                # 이미지 한 장: (1, 3, 2)
x = image.flatten(start_dim=1)              # 픽셀을 한 줄로 펼침: (1, 6)
y = model(x)                                # 모델 출력: (1, 6)
print(x.shape, y.shape, y.view(1, 2, 3).shape)

Linear    70개
Linear    88개
Linear    54개
Sigmoid    0개
전체: 212
torch.Size([1, 6]) torch.Size([1, 6]) torch.Size([1, 2, 3])


**결과 확인: 파이토치로 센 값도 212개**
- 손으로 계산한 70, 88, 54와 계층별 값이 같고 전체도 212개
- 입력 (1, 3, 2)를 (1, 6)으로 펼쳐 모델에 넣고, 출력 (1, 6)을 (1, 2, 3)으로 접어 2 x 3 이미지로 만듦

## 연습문제 3-2

> [그림 3-7]의 다층 퍼셉트론 모델에 입력 (-1, -1), (-1, 1), (1, -1), (1, 1)을 각각 넣었을 때 모델의 출력을 순전파로 계산해 보자.
> 모든 뉴런의 편향은 1이며 가중치는 뉴런 내부에 표시되어 있다.
> 활성화 계층은 입력을 그대로 내보내는 항등 함수라고 가정하자.

**그림의 값을 `nn.Linear`에 넣기**
- 은닉층: 뉴런 3개, 가중치 (1, 1), (-1, 1), (2, 0), 편향 모두 1
- 출력층: 뉴런 2개, 가중치 (1, 1, -1), (-1, 1, 2), 편향 모두 1
- 활성화 계층이 항등 함수이므로 계층 둘만 이어 붙이면 됨
- 네 입력을 (4, 2) 행렬로 묶어 한 번에 순전파
- `copy_(텐서)`는 가중치에 그 텐서의 값을 복사해 넣고, `fill_(값)`은 모든 요소를 그 값으로 채움: 이름 끝의 `_`는 새 텐서를 만들지 않고 값만 바꾸는 제자리 연산 메서드(`zero_()`와 같은 규칙)이며, 파라미터는 `torch.no_grad()` 안에서 바꿈

In [5]:
X = torch.tensor([[-1., -1.], [-1., 1.], [1., -1.], [1., 1.]])    # (4, 2)
hidden = nn.Linear(2, 3)
output = nn.Linear(3, 2)
with torch.no_grad():                       # [그림 3-7]의 값을 그대로 넣음
    hidden.weight.copy_(torch.tensor([[1., 1.], [-1., 1.], [2., 0.]]))
    hidden.bias.fill_(1.)
    output.weight.copy_(torch.tensor([[1., 1., -1.], [-1., 1., 2.]]))
    output.bias.fill_(1.)
    H = hidden(X)                           # 은닉층 출력: (4, 3)
    Y = output(H)                           # 최종 출력: (4, 2)

def ints(values):                           # 소수 없이 정수로 표시
    return tuple(int(v) for v in values)

for x, h, y in zip(X.tolist(), H.tolist(), Y.tolist()):
    print(f'입력 {ints(x)}  은닉 {ints(h)}  출력 {ints(y)}')

입력 (-1, -1)  은닉 (-1, 1, -1)  출력 (2, 1)
입력 (-1, 1)  은닉 (1, 3, -1)  출력 (6, 1)
입력 (1, -1)  은닉 (1, -1, 3)  출력 (-2, 5)
입력 (1, 1)  은닉 (3, 1, 3)  출력 (2, 5)


**결과 분석: 출력 (y1, y2)는 (2, 1), (6, 1), (-2, 5), (2, 5)**
- 입력 (-1, -1)의 은닉층 출력은 (-1, 1, -1)
    - 첫 번째 뉴런: 1 * (-1) + 1 * (-1) + 1 = -1
    - 출력 y1: 1 * (-1) + 1 * 1 + (-1) * (-1) + 1 = 2
    - 출력 y2: (-1) * (-1) + 1 * 1 + 2 * (-1) + 1 = 1
- 나머지 세 입력도 같은 방식으로 계산하며, 코드의 결과와 같음

**계층 둘을 계층 하나로 합치기**
- 활성화가 항등 함수이면 출력 = 출력층 가중치 * (은닉층 가중치 * 입력 + 은닉층 편향) + 출력층 편향
- 이 식은 하나의 선형식: 가중치 W = 출력층 가중치 * 은닉층 가중치, 편향 b = 출력층 가중치 * 은닉층 편향 + 출력층 편향
- 합친 계층 하나로 계산한 결과가 두 계층의 결과와 같은지 확인

In [6]:
with torch.no_grad():
    W = output.weight @ hidden.weight                   # (2, 3) @ (3, 2) = (2, 2)
    b = output.weight @ hidden.bias + output.bias       # (2,)
    Y_single = X @ W.T + b                              # 계층 하나로 계산
print('합친 가중치', W.tolist(), '편향', b.tolist())
print('두 계층의 결과와 같은가:', torch.allclose(Y, Y_single))

합친 가중치 [[-2.0, 2.0], [2.0, 0.0]] 편향 [2.0, 3.0]
두 계층의 결과와 같은가: True


**결과 분석: 계층을 아무리 쌓아도 활성화가 항등 함수이면 선형 계층 하나와 같음**
- 합친 가중치는 ((-2, 2), (2, 0)), 편향은 (2, 3): 계층 하나로 계산한 결과가 두 계층의 결과와 같음
- 활성화가 없으면 계층을 더 쌓아도 표현할 수 있는 함수가 늘지 않음
- 그래서 다층 퍼셉트론은 은닉층 뒤에 시그모이드 같은 비선형 활성화 계층을 둠

**직접 해 보기: 은닉층 뒤에 시그모이드 활성화 함수를 추가하면?**
- 같은 가중치에서 은닉층 출력에 `nn.Sigmoid()`를 적용: 값이 0과 1 사이로 바뀜
- 출력이 달라질까? 계층 하나로 합쳐도 같은 결과가 나올까?

In [7]:
activation = nn.Sigmoid()                               # 시그모이드 활성화 계층
with torch.no_grad():
    Y_sig = output(activation(hidden(X)))               # 은닉층 뒤에 시그모이드
for x, y in zip(X.tolist(), Y_sig.tolist()):
    print(f'입력 {ints(x)}  출력 ({y[0]:.2f}, {y[1]:.2f})')

same = [torch.allclose(a, b) for a, b in zip(Y_sig, Y_single)]
print('계층 하나로 합친 결과와 같은 입력:', same)

입력 (-1, -1)  출력 (1.73, 2.00)
입력 (-1, 1)  출력 (2.41, 1.76)
입력 (1, -1)  출력 (1.05, 2.44)
입력 (1, 1)  출력 (1.73, 2.68)
계층 하나로 합친 결과와 같은 입력: [False, False, False, False]


**결과 확인: 시그모이드를 은닉층 뒤에 추가하면 계층을 하나로 합칠 수 없음**
- 출력은 (1.73, 2.00), (2.41, 1.76), (1.05, 2.44), (1.73, 2.68)로 항등 함수일 때와 네 입력 모두 달라짐
- 시그모이드가 은닉층 출력을 0과 1 사이로 눌러, 입력이 커진 만큼 출력이 일정하게 커지던 선형 관계가 깨짐
- 입력에 따라 다르게 동작하는 비선형 함수가 되어, 계층 하나로는 표현할 수 없음